# Restoring the IIoT experiment

Prem Armstrong D · ECS750P

This notebook is a new executable entry point assembled in October 2026. It is not the original 2024 `.ipynb`. The original pasted code and outputs are preserved separately under `original/`. See `docs/RESTORATION.md` for the changes.

## What I recovered

Three project conversations, including the long user-posted workflow and its results. The exact feature expression was `tcp.ack / (tcp.len + 1)`. The saved outputs reported 814 removed duplicates, 156,986 rows, and a test confusion matrix of `[[4942, 0], [0, 26456]]`.

In [ ]:
from pathlib import Path
import json
root = Path.cwd()
if not (root / "restored_workflow.py").exists():
    raise RuntimeError("Open this notebook from the iiot-traffic-lab folder.")
print((root / "original/user-full-project-paste.txt").read_text()[:700])

## Inspect the fresh replay

These results were measured in this rebuild. They are separate from the historical values pasted into the old chat.

In [ ]:
historical_replay = json.loads((root / "evidence/restored_results.json").read_text())
for key in ["rows", "duplicates_removed", "features", "test_rows", "accuracy", "confusion_matrix"]:
    print(key, historical_replay[key])

## Rerun the restored workflow

Download the authors' `ML-EdgeIIoT-dataset.csv` into `data/` before executing the next cell. The restoration retains whole-dataset preprocessing for historical comparison. It is not the preferred generalisation estimate.

In [ ]:
from restored_workflow import restore
replayed_results = restore()

## Compare a stricter experiment

The separate pipeline excludes identities, timestamps, topics and attack annotations; fits the imputer on training rows only; and keeps identical numeric input vectors together. This controls some leakage routes, but does not establish session, time or independent-site generalisation.

In [ ]:
controlled = json.loads((root / "evidence/results.json").read_text())
for name, metrics in controlled["models"].items():
    print(name, "accuracy:", round(metrics["accuracy"], 4), "balanced accuracy:", round(metrics["balanced_accuracy"], 4), "attack F1:", round(metrics["f1"], 4))

## Demonstrate it

Run `python server.py` in a terminal, then open `http://127.0.0.1:8765`. The interface uses the separately controlled model. It can predict the supplied sample CSV without downloading the full training dataset.

The final next test should use a different traffic capture or site. No energy savings are claimed because this dataset contains no measured power target for this experiment.